# Citi Velocity swap spreads

Citi publishes a swap spread of its own. This repo also *computes* two —
`IRS_MMSS` and `IRS_SPREADOVER`, from ERIS/SDR data against UST CUSIPs — and they
are **different numbers from different inputs**. The Citi one is added alongside
rather than replacing them.

`IRS_CITIVELO_SWAP_SPREAD` is deliberately not named `..._MMSS`: `BT/signals/
tfp_swap_spread.py` selects columns with `"MMSS" in c`, so a name containing it
would have been swept into that backtest's panel silently.

**Units: basis points**, measured 2026-08-08 rather than assumed — USD_SOFR reads
2Y −14.56, 10Y −41.78, 30Y −75.11, the right magnitude, sign and term structure,
three orders of magnitude from a decimal reading.

**Against the repo's own `SPREADOVER`**, median difference over 2026-07-08..08-07:

| 2Y | 3Y | 5Y | 7Y | 10Y | 20Y | 30Y |
|---|---|---|---|---|---|---|
| +0.036 | +0.386 | +0.015 | +0.067 | −0.002 | −0.069 | −0.156 |

Under 0.4 bp on all seven, under 0.1 bp on five — two independent constructions
agreeing. Do **not** quote the mean over those days: the repo's `SPREADOVER`
fails to price on 9 of 23 days and returns values like −151,276 bp when it does,
which no mean survives.

In [1]:
%load_ext autoreload
%autoreload 2
%reload_ext autoreload

import nest_asyncio
nest_asyncio.apply()

import datetime
import pytz
NYC_tz = pytz.timezone("America/New_York")

import warnings
warnings.filterwarnings(
    "ignore",
    category=UserWarning,
)

import sys
sys.path.append("../../")

import pandas as pd
from RVUtils.plt_timeseries import make_secondary_axis_plot

In [2]:
from MDP.IRSwaps.IRSwapsMDP import IRSwapsMDP
from TB.IRSwapsTB import IRSwapsTB
from TB.TimeseriesBuilder import TimeseriesBuilder
from Query.Unified.UnifiedQuery import UnifiedQuery
from Query.Unified.registry import UnifiedValue

irs_mdp = IRSwapsMDP(source="CITIVELO_EXCEL-RL")
ts_builder = TimeseriesBuilder()

## The axis is ragged, so read it from the catalog

`USD_SOFR` and `USD_FEDFUND` carry **eleven** tenors — including money-market
ones, and with no 4Y/15Y/25Y. `GBP_SONIA` carries a different ten (no 1M–1Y, but
15Y/40Y/50Y). **`EUR_EUROSTR` has no `SWAP_SPREAD` node at all.** Only 13 of the
20 OIS indices carry the family, so a hardcoded tenor tuple is wrong off USD.

In [3]:
from MDP.IRSwaps.CITIVELO_EXCEL.swap_spreads import (
    indices_with_swap_spread, swap_spread_tenors,
)

for idx in indices_with_swap_spread():
    print(f"{idx:<18} {', '.join(swap_spread_tenors(idx))}")

AUD_AONIA          2Y, 3Y, 5Y, 7Y, 10Y, 20Y, 30Y
CAD_CORRA          2Y, 5Y, 10Y, 30Y
CHF_SARON          2Y, 5Y, 10Y, 30Y
DKK_TNDKK          2Y, 5Y, 10Y, 30Y
GBP_SONIA          2Y, 3Y, 5Y, 7Y, 10Y, 15Y, 20Y, 30Y, 40Y, 50Y
JPY_TONAR          3M, 6M, 1Y, 2Y, 3Y, 5Y, 7Y, 10Y, 15Y, 20Y, 30Y, 40Y
JPY_TONAR_JSCC     3M, 6M, 1Y, 2Y, 3Y, 5Y, 7Y, 10Y, 15Y, 20Y, 30Y, 40Y
JPY_TONAR_LCH      3M, 6M, 1Y, 2Y, 3Y, 5Y, 7Y, 10Y, 15Y, 20Y, 30Y, 40Y
NOK_NOWA           2Y, 5Y, 10Y
NZD_NZIONA         2Y, 5Y, 7Y, 10Y, 20Y
SEK_STINA          2Y, 5Y, 10Y, 30Y
USD_FEDFUND        1M, 3M, 6M, 1Y, 2Y, 3Y, 5Y, 7Y, 10Y, 20Y, 30Y
USD_SOFR           1M, 3M, 6M, 1Y, 2Y, 3Y, 5Y, 7Y, 10Y, 20Y, 30Y


## Citi's published spread, per tenor

In [4]:
start = datetime.date(2026, 7, 20)
end   = datetime.date(2026, 8, 7)

tenors = swap_spread_tenors("USD_SOFR")
queries = [
    UnifiedQuery(curve="USD-SOFR-1D", tenor=t, value=UnifiedValue.IRS_CITIVELO_SWAP_SPREAD)
    for t in tenors
]

df = ts_builder.get_timeseries(
    start=start,
    end=end,
    queries=queries,
    routers={"IRS": IRSwapsTB(irs_mdp, show_tqdm=True)},
    n_jobs=12,
    ignore_cache_miss=True,
)
df

DuckDB unavailable (file locked), falling back to parquet-only: C:\Users\chris\clee\ARBS\data\ts\computed_ts.duckdb


SUCCESS: `func_tol` reached after 6 iterations (levenberg_marquardt), `f_val`: 4.2675157286224645e-13, `time`: 0.1679s


PRICING USD-SOFR-1D IRSWAPS [workers=12]...:   0%|          | 0/165 [00:00<?, ?it/s]c:\Users\chris\anaconda3\envs\stir\Lib\site-packages\rateslib\data\fixings.py:3426: RuntimeWarning: invalid value encountered in divide
  r = (v[:-1] / v[1:] - 1) * 100 / dcfs_obs[len(populated) :]


,USD-SOFR-1D 10Y OUTRIGHT CITIVELO_SWAP_SPREAD,USD-SOFR-1D 1M OUTRIGHT CITIVELO_SWAP_SPREAD,USD-SOFR-1D 1Y OUTRIGHT CITIVELO_SWAP_SPREAD,USD-SOFR-1D 20Y OUTRIGHT CITIVELO_SWAP_SPREAD,USD-SOFR-1D 2Y OUTRIGHT CITIVELO_SWAP_SPREAD,USD-SOFR-1D 30Y OUTRIGHT CITIVELO_SWAP_SPREAD,USD-SOFR-1D 3M OUTRIGHT CITIVELO_SWAP_SPREAD,USD-SOFR-1D 3Y OUTRIGHT CITIVELO_SWAP_SPREAD,USD-SOFR-1D 5Y OUTRIGHT CITIVELO_SWAP_SPREAD,USD-SOFR-1D 6M OUTRIGHT CITIVELO_SWAP_SPREAD,USD-SOFR-1D 7Y OUTRIGHT CITIVELO_SWAP_SPREAD
Date,,,,,,,,,,,
2026-07-20,-42.1250,6.52967,6.029190,-71.9158,-15.2031,-75.3750,4.22798,-21.0663,-28.7000,2.244040,-37.1629
2026-07-21,-41.8450,7.86158,7.130480,-71.5278,-14.8935,-74.8750,0.23072,-20.4233,-28.5200,-1.633780,-37.0639
2026-07-22,-42.1250,9.82765,8.753610,-72.2606,-14.7663,-75.1250,6.22650,-20.7775,-28.6250,-0.234243,-37.0460
2026-07-23,-43.2469,6.51998,7.826860,-74.3259,-14.7884,-76.7500,2.28261,-20.8449,-29.4250,-3.297710,-38.0827
2026-07-24,-42.6250,-7.99321,9.015670,-73.1712,-15.3673,-76.0000,3.11562,-21.2211,-29.0300,-0.643803,-37.2440
2026-07-27,-42.1250,6.82937,9.003880,-72.6266,-14.9921,-75.4500,3.29657,-21.1516,-28.6750,0.297715,-36.6964
2026-07-28,-41.6250,10.61640,8.344630,-71.4797,-15.2477,-74.7500,-2.64213,-20.7716,-28.7500,1.446310,-36.4742
2026-07-29,-41.5000,5.85629,5.868380,-71.7220,-15.1308,-74.6970,1.01320,-21.0808,-28.7000,0.896935,-36.9960
2026-07-30,-41.5625,6.12161,8.053530,-72.7474,-14.2745,-76.0000,3.39787,-20.1637,-28.5000,2.869510,-37.0357


In [5]:
# The term structure on the last available date
last = df.dropna(how="all").iloc[-1]
last.index = [c.split()[-2] if len(c.split()) > 2 else c for c in last.index]
last.to_frame("swap spread (bp)")

,swap spread (bp)
OUTRIGHT,-40.950000
OUTRIGHT,-10.344300
OUTRIGHT,-0.540213
OUTRIGHT,-71.804900
OUTRIGHT,-14.655700
OUTRIGHT,-74.625000
OUTRIGHT,2.231770
OUTRIGHT,-20.718700
OUTRIGHT,-28.125000
OUTRIGHT,-7.357510


## Citi's number vs the repo's computed one

`scripts/citivelo_swap_spread_tieout.py` does this properly — it excludes the days
the repo side failed to price, counts them, and persists the per-day series so the
meaningless mean cannot be quoted by accident.

In [ ]:
spreadover = [
    UnifiedQuery(curve="USD-SOFR-1D", tenor=t, value=UnifiedValue.IRS_SPREADOVER)
    for t in ("5Y", "10Y", "30Y")
]
citi = [
    UnifiedQuery(curve="USD-SOFR-1D", tenor=t, value=UnifiedValue.IRS_CITIVELO_SWAP_SPREAD)
    for t in ("5Y", "10Y", "30Y")
]

# NOTE the routers. Citi's published spread needs only the IRS router -- it is a
# quote. The repo's SPREADOVER is swap MINUS cash, so it needs an FRB router too
# and raises "IRS/FRB routers must be registered" without one. That difference IS
# the difference between the two numbers, made concrete.
from MDP.FixedRateBonds.FixedRateBondsMDP import FixedRateBondsMDP
from TB.FixedRateBondsTB import FixedRateBondsTB

usts_mdp = FixedRateBondsMDP(source="USTS_FEDINVEST_WSJ_LIVE-QL")

both = ts_builder.get_timeseries(
    start=start,
    end=end,
    queries=citi + spreadover,
    routers={
        "IRS": IRSwapsTB(irs_mdp, show_tqdm=True),
        "FRB": FixedRateBondsTB(usts_mdp, show_tqdm=True),
    },
    n_jobs=12,
    ignore_cache_miss=True,
)
both